# SMS Spam Training and Ratio Experiment

This notebook reproduces the workflow in `train.py`: it cleans the SMS data, compares the original training distribution with five ham-to-spam ratios, evaluates each model on one shared held-out test set, writes `spam_ratio_experiment_report.txt`, and saves the `3600:3600` model as `sms_spam_pipeline.pkl`.

**Oversampling note:** the training split has fewer than 3,600 unique spam messages. The fixed-ratio experiments sample spam with replacement, so repeated spam messages appear in those experiment datasets. No new messages are added to the CSV.

## 1. Imports, text-cleaning setup, and experiment settings

In [1]:
import os
import re
import string

import joblib
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.utils import resample

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

stop_words = set(stopwords.words('english'))
lemm = WordNetLemmatizer()

HAM_SAMPLE_COUNT = 3600
SPAM_SAMPLE_COUNTS = (1200, 1800, 2400, 3000, 3600)
REPORT_FILE = 'spam_ratio_experiment_report.txt'
MODEL_FILE = 'sms_spam_pipeline.pkl'

### Text-cleaning function

The next code cell defines `clean_text`. It lowercases each message, removes URLs, email addresses, digits, and punctuation, removes English stop words, and lemmatizes the remaining words. The data-preparation cell below applies this function to every message.

In [2]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = [
        lemm.lemmatize(word)
        for word in text.split()
        if word not in stop_words
    ]
    return ' '.join(tokens)

## 2. Load, clean, and split the data

The notebook uses `spam.csv` if it exists; otherwise it falls back to `SMS.csv`. It makes one stratified 80/20 split and keeps the test portion aside for all experiments.

In [3]:
csv_file = 'spam.csv' if os.path.exists('spam.csv') else 'SMS.csv'
try:
    df = pd.read_csv(csv_file, encoding='latin-1')
except Exception:
    df = pd.read_csv(csv_file, encoding='utf-8')

if 'v1' in df.columns and 'v2' in df.columns:
    df = df[['v1', 'v2']]
    df.columns = ['label', 'message']

df = df.drop_duplicates().reset_index(drop=True)
df['clean_message'] = df['message'].apply(clean_text)
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_message'],
    df['label'],
    test_size=0.20,
    random_state=42,
    stratify=df['label'],
)

print(f'Dataset: {csv_file}')
print('Training label counts (0=ham, 1=spam):', y_train.value_counts().to_dict())
print('Test label counts (0=ham, 1=spam):', y_test.value_counts().to_dict())

Dataset: spam.csv
Training label counts (0=ham, 1=spam): {0: 3613, 1: 522}
Test label counts (0=ham, 1=spam): {0: 903, 1: 131}


## 3. Create the training sets to compare

The original training distribution is included as a baseline. For each fixed ratio, 3,600 ham messages are sampled without replacement and the requested number of spam entries are sampled with replacement.

In [4]:
train_df = pd.DataFrame({'clean_message': X_train, 'label': y_train})
train_ham = train_df[train_df.label == 0]
train_spam = train_df[train_df.label == 1]

if len(train_ham) < HAM_SAMPLE_COUNT:
    raise ValueError(
        f'The training split has {len(train_ham)} ham messages; '
        f'{HAM_SAMPLE_COUNT} are required for the ratio experiment.'
    )

experiment_sets = [('OG:OG (original distribution)', train_df)]
sampled_ham = resample(
    train_ham,
    replace=False,
    n_samples=HAM_SAMPLE_COUNT,
    random_state=42,
)

for spam_count in SPAM_SAMPLE_COUNTS:
    sampled_spam = resample(
        train_spam,
        replace=True,
        n_samples=spam_count,
        random_state=42,
    )
    experiment_sets.append(
        (
            f'{HAM_SAMPLE_COUNT}:{spam_count}',
            pd.concat([sampled_ham, sampled_spam]).reset_index(drop=True),
        )
    )

print(f'Prepared {len(experiment_sets)} experiment datasets:')
for ratio, experiment_df in experiment_sets:
    print(f'  {ratio}: {experiment_df.label.value_counts().to_dict()}')

Prepared 6 experiment datasets:
  OG:OG (original distribution): {0: 3613, 1: 522}
  3600:1200: {0: 3600, 1: 1200}
  3600:1800: {0: 3600, 1: 1800}
  3600:2400: {0: 3600, 1: 2400}
  3600:3000: {0: 3600, 1: 3000}
  3600:3600: {0: 3600, 1: 3600}


## 4. Train, tune, and evaluate each experiment

For each training set, five-fold cross-validation selects the best SVM `C` value from 0.1, 1, and 10. The selected model is then evaluated against the same untouched test set.

In [5]:
param_grid = {'svm__C': [0.1, 1, 10]}
results = []
final_model = None

for ratio, experiment_df in experiment_sets:
    X_experiment = experiment_df['clean_message']
    y_experiment = experiment_df['label']

    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
        ('svm', LinearSVC(random_state=42)),
    ])
    grid = GridSearchCV(pipeline, param_grid, cv=5)
    grid.fit(X_experiment, y_experiment)

    best_model = grid.best_estimator_
    pred = best_model.predict(X_test)
    tn, ham_as_spam, spam_as_ham, tp = confusion_matrix(
        y_test, pred, labels=[0, 1]
    ).ravel()
    result = {
        'ratio': ratio,
        'train_counts': y_experiment.value_counts().to_dict(),
        'best_params': grid.best_params_,
        'cv_accuracy': grid.best_score_,
        'accuracy': accuracy_score(y_test, pred),
        'ham_as_spam': int(ham_as_spam),
        'spam_as_ham': int(spam_as_ham),
        'precision': precision_score(y_test, pred),
        'recall': recall_score(y_test, pred),
        'f1': f1_score(y_test, pred),
        'classification_report': classification_report(y_test, pred),
    }
    results.append(result)

    print(
        f"\n{ratio} (ham:spam) | "
        f"train={result['train_counts']} | "
        f"accuracy={result['accuracy']:.4f} | "
        f"precision={result['precision']:.4f} | "
        f"recall={result['recall']:.4f} | "
        f"f1={result['f1']:.4f}"
    )
    print('Best Parameters:', grid.best_params_)

    if ratio == f'{HAM_SAMPLE_COUNT}:{SPAM_SAMPLE_COUNTS[-1]}':
        final_model = best_model

if final_model is None:
    raise RuntimeError('The 3600:3600 experiment did not produce a model.')


OG:OG (original distribution) (ham:spam) | train={0: 3613, 1: 522} | accuracy=0.9816 | precision=0.9590 | recall=0.8931 | f1=0.9249
Best Parameters: {'svm__C': 1}

3600:1200 (ham:spam) | train={0: 3600, 1: 1200} | accuracy=0.9797 | precision=0.9365 | recall=0.9008 | f1=0.9183
Best Parameters: {'svm__C': 10}

3600:1800 (ham:spam) | train={0: 3600, 1: 1800} | accuracy=0.9797 | precision=0.9435 | recall=0.8931 | f1=0.9176
Best Parameters: {'svm__C': 1}

3600:2400 (ham:spam) | train={0: 3600, 1: 2400} | accuracy=0.9807 | precision=0.9302 | recall=0.9160 | f1=0.9231
Best Parameters: {'svm__C': 1}

3600:3000 (ham:spam) | train={0: 3600, 1: 3000} | accuracy=0.9768 | precision=0.9213 | recall=0.8931 | f1=0.9070
Best Parameters: {'svm__C': 10}

3600:3600 (ham:spam) | train={0: 3600, 1: 3600} | accuracy=0.9768 | precision=0.9213 | recall=0.8931 | f1=0.9070
Best Parameters: {'svm__C': 10}


## 5. Save the experiment report and model

The report includes detailed classification metrics and an overall comparison table. The saved model is the best estimator selected within the `3600:3600` experiment, matching `train.py`.

In [6]:
report_lines = [
    'SMS Spam SVM - Ham/Spam Training Ratio Experiment',
    '=' * 54,
    f'Dataset: {csv_file}',
    'Split: stratified 80% train / 20% test, random_state=42',
    'Test set: same untouched holdout for every experiment',
    f'Training split original counts (label 0=ham, 1=spam): {y_train.value_counts().to_dict()}',
    f'Test split counts (label 0=ham, 1=spam): {y_test.value_counts().to_dict()}',
    '',
    'Holdout metrics by training ratio (ham:spam):',
]

for result in results:
    report_lines.extend([
        '',
        f"Ratio: {result['ratio']}",
        '  Training counts (ham:spam): '
        f"{result['train_counts'].get(0, 0)}:{result['train_counts'].get(1, 0)}",
        f"  Best SVM parameters: {result['best_params']}",
        f"  5-fold CV accuracy: {result['cv_accuracy']:.4f}",
        f"  Test accuracy: {result['accuracy']:.4f}",
        f"  Test precision: {result['precision']:.4f}",
        f"  Test recall: {result['recall']:.4f}",
        f"  Test F1-score: {result['f1']:.4f}",
        '  Classification report:',
        *[f'    {line}' for line in result['classification_report'].rstrip().splitlines()],
    ])

report_lines.extend([
    '',
    f'Saved model: {MODEL_FILE}',
    f'Saved model training ratio: {HAM_SAMPLE_COUNT}:{SPAM_SAMPLE_COUNTS[-1]} (ham:spam)',
    '',
    'FINAL COMPARISON - HOLDOUT TEST RESULTS',
    'Ratio (ham:spam)             Accuracy   Ham reported as spam   Spam reported as ham',
    '----------------------------  --------   --------------------   --------------------',
])

for result in results:
    report_lines.append(
        f"{result['ratio']:<28}  "
        f"{result['accuracy'] * 100:>6.2f}%"
        f"{result['ham_as_spam']:>22}"
        f"{result['spam_as_ham']:>23}"
    )

report_lines.extend([
    '',
    'Ham reported as spam = ham messages incorrectly classified as spam.',
    'Spam reported as ham = spam messages incorrectly classified as ham.',
    '',
])

with open(REPORT_FILE, 'w', encoding='utf-8') as report_file:
    report_file.write('\n'.join(report_lines))

joblib.dump(final_model, MODEL_FILE)
print(f'Experiment report saved to {REPORT_FILE}')
print(f'Model saved to {MODEL_FILE}')

Experiment report saved to spam_ratio_experiment_report.txt
Model saved to sms_spam_pipeline.pkl
